In [1]:
import uuid, time
from src.config.parameters_config import get_notebook_parameters
from src.config.gold_config import GoldConfig
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.reader import DataFrameReader
from src.io.writer import DataFrameWriter

In [ ]:
layer = "gold"

# Load Spark Session
spark = get_spark_session()
dataset = get_notebook_parameters()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Load Gold config
gold_cfg = GoldConfig(dataset, layer)
gold_cfg.load_yaml()

In [3]:
# Create variables from yaml
source_table = gold_cfg.source_table
source_schema = gold_cfg.source_schema
source_catalog = gold_cfg.source_catalog
target_table = gold_cfg.target_table
target_schema = gold_cfg.target_schema
target_catalog = gold_cfg.target_catalog

# Create logical variables
run_id = str(uuid.uuid4())

In [4]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")
log.info(f"[CTX] source_table={source_table} target_table={target_table}")
log.info(f"[CTX] target_schema={target_schema} run_id={run_id}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:
    reader = DataFrameReader(spark)
    writer = DataFrameWriter(spark)
    
    # Read table from Silver
    df_gold = reader.read_table(
        catalog=source_catalog, 
        schema=source_schema, 
        table=source_table
    )
    log.info(f"[STEP 1] READ: Silver table loaded: {source_table} rows")

    select_columns = gold_cfg.select_columns
    df_gold_selected = df_gold.select(*select_columns)
    log.info(f"[STEP 2] SELECT: Filter columns from Silver table: {select_columns}")

    writer.write_delta_batch(
        df=df_gold_selected,
        catalog=target_catalog,
        schema=target_schema,
        table=target_table,
        mode="overwrite"
    )
    
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise